In [0]:
%sql
drop table `p-cdh`.reference.dwh_stg_sme_re_collateral_strategy

In [0]:
spark.sql(f'''CREATE TABLE `p-cdh`.reference.sme_re_collateral_strategy
          AS SELECT * FROM `d-cdh`.reference.dwh_stg_sme_re_collateral_strategy
''')

In [0]:
# Create view for DAgreement 
create_view =f"""
create or replace view `p-cdh`.derivative.dagreement
as SELECT * FROM `p-cdh`.derivativeactive.dagreement
          """
spark.sql(create_view)

In [0]:
min_date = "2025-02-03"
max_date = "2025-02-26"

# Run the Spark SQL query
result = spark.sql(f"""
    SELECT DISTINCT businessdate  
    FROM `p-cdh`.primaryactive.portfolio  
    WHERE businessdate BETWEEN '{min_date}' AND '{max_date}'  
    ORDER BY businessdate
""").collect()

# Extract values into a list
business_dates = [row["businessdate"] for row in result]

# Print the collected dates
print(business_dates)


In [0]:
for date in business_dates:
    ## Update GBVInitializationDate in portfolio for Domus & Rhodium 
    spark.sql(f"""
    UPDATE `p-cdh`.primaryactive.portfolio
    SET GBVInitializationDate = '2025-02-04'
    WHERE portfolioname LIKE '%DOMUS%' OR portfolioname LIKE '%RHODIUM%'
    AND BusinessDate = '{date}'
    """)

In [0]:

spark.sql(f"""
MERGE INTO `p-cdh`.primaryactive.party p
USING `p-cdh`.landed.profits_customer c
on p.businessdate = c.businessdate
and c.CUST_ID = p.PartyKey
and p.sourcesystem = 'profits'
WHEN MATCHED THEN
UPDATE SET FirstMotherNameGR = MOTHER_NAME
""")


In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.primaryactive.party p
USING `p-cdh`.landed.qca_fs_customers_snap fs
on p.businessdate = fs.businessdate
and fs.cust_code = p.PartyKey
and fs.CUST_SLC_ID = "ALL"
and p.sourcesystem = 'qca'
WHEN MATCHED THEN
UPDATE SET FirstMotherNameGR = CUST_GENERIC_TEXT_01
""")

In [0]:
spark.sql(f"""
MERGE INTO `p-cdh`.primaryactive.party p
USING `p-cdh`.landed.arotron_eaiSIG_BP e
on p.businessdate = e.businessdate
and e.eaisig_bp_cust_s_unid = p.PartyKey
and p.sourcesystem = 'arotron'
WHEN MATCHED THEN
UPDATE SET FirstMotherNameGR = eaisig_bp_cust_mothname
""")

In [0]:
spark.sql("""
MERGE INTO `p-cdh`.derivativeactive.dparty dp
USING (
    select distinct businessdate, PartyKey, sourcesystem, FirstMotherNameGR
    from `p-cdh`.primaryactive.party
    where PartyType <> 'Organization'
    ) p
on p.businessdate = dp.businessdate
and p.PartyKey = dp.PartyKey
and p.sourcesystem = dp.sourcesystem
and dp.PartyType <> 'Organization'
WHEN MATCHED THEN
UPDATE SET dp.FirstMotherNameGR = p.FirstMotherNameGR
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE `p-cdh`.reference.lookup_export 
AS SELECT * FROM `qa-cdh`.reference.lookup_export 
""")